# Первый черновик модели риска неплатежа

Цель — сравнить простое правило по последнему платёжному статусу
с логистической регрессией на исходных и нескольких понятных
производных признаках.

Работаем только с train.csv.

Тестовая выборка не используется на этапе разработки.

In [1]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

TARGET = "default payment next month"
RANDOM_STATE = 42

In [2]:
uploaded = files.upload()

In [3]:
train = pd.read_csv("train.csv")

train.shape

(24000, 25)

In [4]:
train_summary = pd.Series({
    "Клиентов": len(train),
    "Событий неплатежа": train[TARGET].sum(),
    "Доля событий, %": train[TARGET].mean() * 100,
    "Уникальных ID": train["ID"].nunique()
})

train_summary

Клиентов             24000.000000
Событий неплатежа     5309.000000
Доля событий, %         22.120833
Уникальных ID        24000.000000
dtype: float64

In [5]:
payment_status_columns = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

train["TOTAL_BILL_AMT"] = train[bill_columns].sum(axis=1)
train["TOTAL_PAY_AMT"] = train[payment_columns].sum(axis=1)
train["NUM_DELINQ"] = (
    train[payment_status_columns] > 0
).sum(axis=1)

In [6]:
train[
    [
        "BILL_AMT1",
        "BILL_AMT2",
        "BILL_AMT3",
        "BILL_AMT4",
        "BILL_AMT5",
        "BILL_AMT6",
        "TOTAL_BILL_AMT"
    ]
].head()

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,TOTAL_BILL_AMT
0,161771,172632,168541,164310,162681,163005,992940
1,0,53,0,0,0,0,53
2,10171,8150,8446,8067,8227,8400,51461
3,206879,211037,215139,218513,131660,134346,1117574
4,0,0,326,326,-235,-235,182


In [7]:
train[
    [
        "PAY_AMT1",
        "PAY_AMT2",
        "PAY_AMT3",
        "PAY_AMT4",
        "PAY_AMT5",
        "PAY_AMT6",
        "TOTAL_PAY_AMT"
    ]
].head()

,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,TOTAL_PAY_AMT
0,15000,0,0,6100,12300,6100,39500
1,53,0,0,0,0,0,53
2,2,1281,1134,294,305,1000,4016
3,9100,9000,7887,4800,4900,6000,41687
4,0,326,0,0,0,1200,1526


In [8]:
train[
    [
        "PAY_0",
        "PAY_2",
        "PAY_3",
        "PAY_4",
        "PAY_5",
        "PAY_6",
        "NUM_DELINQ"
    ]
].head()

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,NUM_DELINQ
0,2,2,3,2,0,0,4
1,1,-1,-1,-2,-2,-2,1
2,1,2,0,0,0,0,2
3,0,0,0,0,0,0,0
4,1,-2,-1,0,0,-2,1


In [9]:
feature_columns = [
    "LIMIT_BAL",
    "AGE",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    *payment_status_columns,
    *bill_columns,
    *payment_columns,
    "TOTAL_BILL_AMT",
    "TOTAL_PAY_AMT",
    "NUM_DELINQ"
]

X = train[feature_columns]
y = train[TARGET]

X.shape

(24000, 26)

In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train.shape, X_valid.shape

((19200, 26), (4800, 26))

In [11]:
validation_summary = pd.DataFrame({
    "Часть": ["Обучение модели", "Валидация"],
    "Клиентов": [len(X_train), len(X_valid)],
    "Событий": [int(y_train.sum()), int(y_valid.sum())],
    "Доля событий, %": [
        y_train.mean() * 100,
        y_valid.mean() * 100
    ]
})

validation_summary.round(2)

,Часть,Клиентов,Событий,"Доля событий, %"
0,Обучение модели,19200,4247,22.12
1,Валидация,4800,1062,22.12


In [12]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numeric_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

numeric_features

['LIMIT_BAL',
 'AGE',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'TOTAL_BILL_AMT',
 'TOTAL_PAY_AMT',
 'NUM_DELINQ']

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [14]:
model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "logistic_regression",
            LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="lbfgs"
            )
        )
    ]
)

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('logistic_regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshol

In [15]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('logistic_regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['LIMIT_BAL','AGE','SEX',...,'TOTAL_BILL_AMT','TOTAL_PAY_AMT','NUM_DELINQ']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [16]:
train_scores = model.predict_proba(X)[:, 1]

train_predictions = train[["ID", TARGET, "PAY_0"]].copy()
train_predictions["model_score"] = train_scores

train_predictions.head(10)

,ID,default payment next month,PAY_0,model_score
0,22789,0,2,0.751600
1,29007,0,1,0.281203
2,16951,0,1,0.379735
3,22281,1,0,0.124226
4,11347,0,1,0.240425
5,4103,1,-1,0.239219
6,28655,0,-1,0.366509
7,26151,1,3,0.902922
8,15092,0,0,0.093830
9,6267,0,0,0.127685


In [17]:
def evaluate_top_share(
    data,
    target_column,
    scores,
    share=0.10,
    method_name=""
):
    selected_count = int(np.ceil(len(data) * share))

    ranked = (
        data
        .assign(score=scores)
        .sort_values(
            ["score", "ID"],
            ascending=[False, True]
        )
        .head(selected_count)
    )

    y = ranked[target_column]
    base_rate = data[target_column].mean()

    return {
        "Метод": method_name,
        "Нагрузка, %": len(ranked) / len(data) * 100,
        "Событий в отборе": int(y.sum()),
        "Доля событий, %": y.mean() * 100,
        "Recall, %": y.sum() / data[target_column].sum() * 100,
        "Lift": y.mean() / base_rate
    }

In [18]:
rule_scores = train["PAY_0"].astype(float)

comparison = pd.DataFrame([
    evaluate_top_share(
        train,
        TARGET,
        rule_scores,
        share=0.10,
        method_name="Правило PAY_0"
    ),
    evaluate_top_share(
        train,
        TARGET,
        train_predictions["model_score"],
        share=0.10,
        method_name="Логистическая регрессия v0"
    )
])

comparison.round(2)

,Метод,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift
0,Правило PAY_0,10.0,1670,69.58,31.46,3.15
1,Логистическая регрессия v0,10.0,1698,70.75,31.98,3.20


In [19]:
valid_scores = model.predict_proba(X_valid)[:, 1]

valid_predictions = train.loc[X_valid.index, ["ID", TARGET, "PAY_0"]].copy()
valid_predictions["model_score"] = valid_scores

valid_predictions.head(10)

,ID,default payment next month,PAY_0,model_score
13927,7837,0,-1,0.239743
19830,24984,0,1,0.411159
23877,13978,0,0,0.103010
14939,17298,0,0,0.117025
22651,23046,0,1,0.266082
12437,27872,0,-1,0.056377
17516,16450,0,-2,0.180449
7390,12945,0,-1,0.273791
21022,24934,0,2,0.722135
21425,3279,0,0,0.291358


In [20]:
assert len(valid_predictions) == 4800
assert valid_predictions[TARGET].sum() == 1062
assert valid_predictions["ID"].nunique() == 4800

In [21]:
rule_scores_valid = valid_predictions["PAY_0"].astype(float)

comparison_valid = pd.DataFrame([
    evaluate_top_share(
        valid_predictions,
        TARGET,
        rule_scores_valid,
        share=0.10,
        method_name="Правило PAY_0"
    ),
    evaluate_top_share(
        valid_predictions,
        TARGET,
        valid_predictions["model_score"],
        share=0.10,
        method_name="Логистическая регрессия v0"
    )
])

comparison_valid.round(2)

,Метод,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift
0,Правило PAY_0,10.0,322,67.08,30.32,3.03
1,Логистическая регрессия v0,10.0,331,68.96,31.17,3.12


In [22]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_valid, valid_scores)

roc_auc

0.7601028879356817

## Оценка на валидационной части

Модель обучена на 19 200 клиентах и оценена на 4 800 клиентах,
которые не участвовали в обучении.

Правило PAY_0 и логистическая регрессия сравниваются на одинаковой
валидационной выборке и при одинаковой нагрузке — верхних 10%
клиентов по ранжированию.

Полученный результат является более содержательным, чем оценка
на train.csv, но всё ещё относится к разработке модели.
Финальная проверка будет выполнена на отдельной тестовой выборке.

## Кросс-валидация top-10%

Цель — проверить, сохраняется ли преимущество логистической
регрессии над правилом PAY_0 при разных разбиениях обучающей
выборки.

Тестовая выборка по-прежнему не используется.

In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

N_SPLITS = 5
TOP_SHARE = 0.10

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = []

for fold, (fit_index, valid_index) in enumerate(cv.split(X, y), start=1):
    X_fold_fit = X.iloc[fit_index]
    y_fold_fit = y.iloc[fit_index]

    X_fold_valid = X.iloc[valid_index]
    y_fold_valid = y.iloc[valid_index]

    fold_model = clone(model)
    fold_model.fit(X_fold_fit, y_fold_fit)

    fold_valid_predictions = train.loc[
        X_fold_valid.index,
        ["ID", TARGET, "PAY_0"]
    ].copy()

    fold_valid_predictions["model_score"] = fold_model.predict_proba(
        X_fold_valid
    )[:, 1]

    rule_result = evaluate_top_share(
        fold_valid_predictions,
        TARGET,
        fold_valid_predictions["PAY_0"].astype(float),
        share=TOP_SHARE,
        method_name="Правило PAY_0"
    )

    model_result = evaluate_top_share(
        fold_valid_predictions,
        TARGET,
        fold_valid_predictions["model_score"],
        share=TOP_SHARE,
        method_name="Логистическая регрессия v0"
    )

    rule_result["Фолд"] = fold
    model_result["Фолд"] = fold

    cv_results.append(rule_result)
    cv_results.append(model_result)

cv_results_df = pd.DataFrame(cv_results)

cv_results_df[
    [
        "Фолд",
        "Метод",
        "Нагрузка, %",
        "Событий в отборе",
        "Доля событий, %",
        "Recall, %",
        "Lift"
    ]
].round(2)

,Фолд,Метод,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift
0,1,Правило PAY_0,10.0,326,67.92,30.73,3.07
1,1,Логистическая регрессия v0,10.0,341,71.04,32.14,3.21
2,2,Правило PAY_0,10.0,344,71.67,32.39,3.24
3,2,Логистическая регрессия v0,10.0,349,72.71,32.86,3.29
4,3,Правило PAY_0,10.0,335,69.79,31.54,3.15
5,3,Логистическая регрессия v0,10.0,333,69.38,31.36,3.14
6,4,Правило PAY_0,10.0,324,67.50,30.51,3.05
7,4,Логистическая регрессия v0,10.0,334,69.58,31.45,3.15
8,5,Правило PAY_0,10.0,338,70.42,31.83,3.18
9,5,Логистическая регрессия v0,10.0,335,69.79,31.54,3.15


In [24]:
cv_summary = (
    cv_results_df
    .groupby("Метод")
    .agg(
        Среднее_событий=("Событий в отборе", "mean"),
        Средний_recall=("Recall, %", "mean"),
        Средний_lift=("Lift", "mean"),
        Минимальный_recall=("Recall, %", "min"),
        Максимальный_recall=("Recall, %", "max")
    )
    .reset_index()
)

cv_summary.round(2)

,Метод,Среднее_событий,Средний_recall,Средний_lift,Минимальный_recall,Максимальный_recall
0,Логистическая регрессия v0,338.4,31.87,3.19,31.36,32.86
1,Правило PAY_0,333.4,31.40,3.14,30.51,32.39


## Вывод по версии v0

Логистическая регрессия v0 в среднем незначительно превосходит
правило PAY_0 при отборе верхних 10% клиентов.

Средний recall модели составляет 31,87% против 31,40% у правила.
Средний lift — 3,19 против 3,14.

Преимущество невелико и не воспроизводится в каждом фолде:
модель лучше правила в трёх фолдах из пяти. Поэтому v0 принимается
как базовая версия для дальнейшего сравнения.

Следующая версия будет проверять, добавляют ли понятные производные
признаки устойчивый прирост качества при той же нагрузке.

## Ограничения первой версии

Оценки на train.csv получены после обучения модели на этой же
выборке. Они показывают работоспособность пайплайна и позволяют
сравнить ранжирования, но не являются оценкой качества на новых
данных.

Для финальной проверки используется отдельная тестовая выборка,
которая не участвовала в исследовании, создании признаков,
выборе правил и обучении модели.

Сравнение правила и модели выполняется при одинаковом размере
отбора — верхних 10% клиентов по ранжированию.